In [1]:
import json

file = json.load(open("data.json", "r"))
print(file[1])

{'document': 'The Delhi High Court on 10th February 2023 held that an employer cannot terminate an employee on maternity leave. The Court directed reinstatement of Ms. Kavita Sharma with full salary and benefits under Section 12 of the Maternity Benefit Act, 1961.', 'summary': 'Delhi High Court said firing a woman during maternity leave is illegal. Ms. Kavita Sharma must be reinstated with full pay and benefits.'}


In [2]:
#!pip install unsloth trl peft accelerate bitsandbytes

In [3]:
!pip uninstall -y pyarrow
!pip install pyarrow==19.0.0

Found existing installation: pyarrow 24.0.0
Uninstalling pyarrow-24.0.0:
  Successfully uninstalled pyarrow-24.0.0
   ---------------------------------------- 0.0/25.3 MB ? eta -:--:--
   ---------------------------------------- 0.3/25.3 MB ? eta -:--:--
   ---- ----------------------------------- 2.6/25.3 MB 9.4 MB/s eta 0:00:03
   -------- ------------------------------- 5.2/25.3 MB 11.0 MB/s eta 0:00:02
   ------------- -------------------------- 8.4/25.3 MB 12.4 MB/s eta 0:00:02
   ------------------- -------------------- 12.3/25.3 MB 13.8 MB/s eta 0:00:01
   --------------------------- ------------ 17.6/25.3 MB 16.0 MB/s eta 0:00:01
   ------------------------------------- -- 23.6/25.3 MB 18.0 MB/s eta 0:00:01
   ---------------------------------------- 25.3/25.3 MB 17.8 MB/s  0:00:01


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.3.0 requires pyarrow>=21.0.0, but you have pyarrow 19.0.0 which is incompatible.


In [4]:
# For GPU check
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [5]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit"

max_seq_length = 2048  # Choose sequence length
dtype = None  # Auto detection

# Load model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


d:\Conda\envs\DL\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0625 10:55:00.870000 36296 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4050 Laptop GPU. Num GPUs = 1. Max memory: 5.997 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 291/291 [00:05<00:00, 51.22it/s]


In [6]:
from datasets import Dataset
import json

def format_prompt(example):
    return f"### Input: {example['document']}\n### Output: {json.dumps(example['summary'])}<|endoftext|>"

formatted_data = [format_prompt(item) for item in file]
dataset = Dataset.from_dict({"text": formatted_data})

In [7]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=64,  # LoRA rank - higher = more capacity, more memory
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=128,  # LoRA scaling factor (usually 2x rank)
    lora_dropout=0,  # Supports any, but = 0 is optimized
    bias="none",     # Supports any, but = "none" is optimized
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized version
    random_state=3407,
    use_rslora=False,  # Rank stabilized LoRA
    loftq_config=None, # LoftQ
)

Unsloth 2026.6.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [9]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Training arguments optimized for Unsloth
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # Effective batch size = 8
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        save_strategy="epoch",
        save_total_limit=2,
        dataloader_pin_memory=False,
        report_to="none", # Disable Weights & Biases logging
    ),
)

Unsloth: Tokenizing ["text"]: 100%|██████████| 81/81 [00:00<00:00, 1927.96 examples/s]


In [10]:
# Train the model
# Align SFTConfig in sys.modules to prevent PicklingError
import sys
if 'trl.trainer.sft_config' in sys.modules and 'trl.trainer.sft_trainer' in sys.modules:
    import trl.trainer.sft_config
    import trl.trainer.sft_trainer
    trl.trainer.sft_config.SFTConfig = trl.trainer.sft_trainer.SFTConfig

trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 81 | Num Epochs = 3 | Total steps = 33
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 119,537,664 of 3,940,617,216 (3.03% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss


Unsloth: Restored added_tokens_decoder metadata in outputs\checkpoint-11\tokenizer_config.json.


PicklingError: Can't pickle <class 'trl.trainer.sft_config.SFTConfig'>: it's not the same object as trl.trainer.sft_config.SFTConfig

In [ ]:
# Test the fine-tuned model
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# Test prompt
messages = [
    {"role": "user", "content": "Extract the product information:\n<div class='product'><h2>iPad Air</h2><span class='price'>$1344</span><span class='category'>audio</span><span class='brand'>Dell</span></div>"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

# Generate response
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    use_cache=True,
    temperature=0.7,
    do_sample=True,
    top_p=0.9,
)

# Decode and print
response = tokenizer.batch_decode(outputs)[0]
print(response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|user|> Extract the product information:
<div class='product'><h2>iPad Air</h2><span class='price'>$1344</span><span class='category'>audio</span><span class='brand'>Dell</span></div><|end|><|assistant|> "Product: iPad Air, Price: $1344, Category: Audio, Brand: Dell"<|end|>


In [ ]:
model.save_pretrained_gguf("gguf_model", tokenizer, quantization_method="q4_k_m")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Unsloth: Preparing safetensor model files:  50%|█████     | 1/2 [03:24<03:24, 204.70s/it]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.65G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [05:05<00:00, 152.62s/it]


Unsloth: Merge process complete. Saved to `/content/gguf_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: All required system packages already installed!
Unsloth: Install llama.cpp and building - please wait 1 to 3 minutes
Unsloth: Cloning llama.cpp repository
Unsloth: Install GGUF and other packages
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['phi-3-mini-4k-instruct.F16.gguf']
Unslot

{'save_directory': 'gguf_model',
 'gguf_files': ['phi-3-mini-4k-instruct.Q4_K_M.gguf'],
 'modelfile_location': '/content/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [ ]:
from google.colab import files
import os

gguf_files = [f for f in os.listdir("gguf_model") if f.endswith(".gguf")]
if gguf_files:
    gguf_file = os.path.join("gguf_model", gguf_files[0])
    print(f"Downloading: {gguf_file}")
    files.download(gguf_file)

In [ ]:
from unsloth import FastLanguageModel

# Enable native 2x faster inference
FastLanguageModel.for_inference(model)

print("Start chatting with the model. Type 'exit' or 'stop' to end the conversation.")

messages = []

while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "stop"]:
        break

    messages.append({"role": "user", "content": user_input})

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        use_cache=True,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
    )

    response = tokenizer.batch_decode(outputs)[0]
    # Extract only the assistant's response
    assistant_response = response.split("<|assistant|>")[-1].strip()
    assistant_response = assistant_response.replace("<|end|>", "").strip()

    print(f"Model: {assistant_response}")

    # Add model's response to the messages for continuity
    messages.append({"role": "assistant", "content": assistant_response})

Start chatting with the model. Type 'exit' or 'stop' to end the conversation.
You: hi what can you help me with
Model: "Hello! I'd be happy to help you. Please tell me what you need assistance with."
You: i want my car insurance
Model: "Sure, I can guide you through the process of claiming your car insurance. First, you'll need to report the incident to your insurance provider. Have you done that yet?"
You: This Agreement is made on 1st March 2024 between Alpha Tech Pvt. Ltd., Bengaluru and Beta Solutions, Pune. Alpha shall deliver 300 laptops within 45 days of signing. Beta agrees to pay INR 45,00,000 within 15 days after delivery. Late payments will attract 10% annual interest. All disputes fall under Bengaluru jurisdiction.
Model: "A contract signed on 1st March 2024 between Alpha Tech in Bengaluru and Beta Solutions in Pune. Alpha will deliver 300 laptops in 45 days. Beta will pay INR 45 lakh within 15 days or face 10% interest. Bengaluru courts handle disputes."
You: exit
